# Mock market data seed generator

Deterministic generator for FMS mock market data. Reuses `TickSimulationEngine`
and `CandleAggregator` (seeded, reproducible) to produce ticks, quotes,
candles, market states, and behavior history for one simulation session, then
writes everything to a portable `.sql` file matching `migrations/` and
`Platform/Trading Platform Schema.sql`.

Scope: FMS market-data tables only (`simulation_sessions`, `stocks`,
`market_states`, `market_behaviors`, `quotes`, `market_ticks`, `candles`). No
users/accounts/orders/instruments are generated — this is not trading
activity.

See [`../data_seed/README.md`](../data_seed/README.md) for the design plan.

## 1. Environment and dependency setup

Import the app package after locating the simulator project root from the
current working directory. This works when Jupyter starts in the repository
root, simulator root, `notebooks/`, or this nested `data_seed/` folder.

In [1]:
import json
import sys
from datetime import datetime, timedelta
from decimal import Decimal
from pathlib import Path

def find_project_root(start: Path) -> Path:
    """Find the nearest parent containing the simulator package and project metadata."""
    for location in (start.resolve(), *start.resolve().parents):
        for candidate in (location, location / "Simulated_Engine"):
            if (candidate / "pyproject.toml").is_file() and (candidate / "app").is_dir():
                return candidate
    raise RuntimeError(
        f"Could not locate the Simulated_Engine project root from {start.resolve()}"
    )


PROJECT_ROOT = find_project_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from app.models.market_data import MarketState, MarketTrend, Quote, Stock
from app.simulation.behaviors import BehaviorType, MarketBehaviorConfig
from app.simulation.candles import CandleAggregator
from app.simulation.clock import MARKET_TIMEZONE
from app.simulation.ticks import TickSimulationConfig, TickSimulationEngine

print(f"app package resolved from: {PROJECT_ROOT}")

app package resolved from: c:\Users\chris\Downloads\Github Desktop\FMS\Simulated_Engine\notebooks\data_seed


## 2. Project configuration and constants

Fixed seed, drift, run length, and the 5 mock stocks. Everything below is
deterministic: re-running this notebook without code changes reproduces
byte-identical output.

In [2]:
SEED = 42
DRIFT = 0.08
STEPS = 300
STEP_SIZE = timedelta(seconds=1)
START_TIME = datetime(2026, 1, 2, 8, 30, tzinfo=MARKET_TIMEZONE)
SESSION_ID = 1  # hardcoded so seed rows are stable across re-runs of this notebook
SESSION_CONFIG = {"steps": STEPS, "step_size_seconds": STEP_SIZE.total_seconds()}

OUTPUT_SQL_PATH = PROJECT_ROOT / "migrations" / "seed_mock_market_data.sql"

STOCKS = [
    Stock(symbol="ACME", company_name="Acme Technologies Inc.", starting_price=Decimal("182.50"),
          sector="Technology", average_volume=8_500_000, base_volatility=Decimal("0.028")),
    Stock(symbol="BLTV", company_name="Bolt Ventures Corp.", starting_price=Decimal("64.10"),
          sector="Financials", average_volume=3_200_000, base_volatility=Decimal("0.019")),
    Stock(symbol="CRSN", company_name="Crestline Energy Co.", starting_price=Decimal("41.75"),
          sector="Energy", average_volume=5_100_000, base_volatility=Decimal("0.033")),
    Stock(symbol="DPHM", company_name="Delphi Health & Sciences", starting_price=Decimal("97.20"),
          sector="Healthcare", average_volume=2_400_000, base_volatility=Decimal("0.021")),
    Stock(symbol="EVRN", company_name="Evergreen Retail Group", starting_price=Decimal("28.90"),
          sector="Consumer Discretionary", average_volume=6_800_000, base_volatility=Decimal("0.024")),
]

[stock.symbol for stock in STOCKS]

['ACME', 'BLTV', 'CRSN', 'DPHM', 'EVRN']

## 3. Input configuration and validation

Behaviors are attached to the engine *before* generation so they actually
influence the produced ticks (not just decorative `market_behaviors` rows).
Pydantic validates each `MarketBehaviorConfig` (symbol pattern, positive
duration) at construction time, so invalid config fails fast here.

In [3]:
BEHAVIOR_CONFIGS = [
    MarketBehaviorConfig(symbol="ACME", behavior_type=BehaviorType.UPTREND,
                          duration=STEP_SIZE * STEPS, strength=0.6),
    MarketBehaviorConfig(symbol="CRSN", behavior_type=BehaviorType.VOLATILITY_SPIKE,
                          duration=STEP_SIZE * STEPS, strength=0.7),
]

known_symbols = {stock.symbol for stock in STOCKS}
unknown_behavior_symbols = {cfg.symbol for cfg in BEHAVIOR_CONFIGS} - known_symbols
if unknown_behavior_symbols:
    raise ValueError(f"behavior configs reference unknown symbols: {sorted(unknown_behavior_symbols)}")

engine = TickSimulationEngine(TickSimulationConfig(seed=SEED, drift=DRIFT))
for behavior_config in BEHAVIOR_CONFIGS:
    engine.add_behavior_from_config(behavior_config, current_time=START_TIME)

[(cfg.symbol, cfg.behavior_type.value, cfg.strength) for cfg in BEHAVIOR_CONFIGS]

[('ACME', 'uptrend', 0.6), ('CRSN', 'volatility_spike', 0.7)]

## 4. Core generation

Generate one global, sequentially-numbered stream of ticks, then derive
quotes, candles, market states, and the behavior history from that single
source of truth.

In [4]:
# One tick per stock per second, globally sequence-numbered by the engine.
ticks = engine.simulate(STOCKS, start_time=START_TIME, steps=STEPS, step_size=STEP_SIZE)
len(ticks)

1500

In [5]:
# Quotes reuse each tick's bid/ask/size snapshot at the same timestamp.
quotes = [
    Quote(symbol=tick.symbol, timestamp=tick.timestamp, bid=tick.bid, ask=tick.ask,
          bid_size=tick.bid_size, ask_size=tick.ask_size)
    for tick in ticks
]
len(quotes)

1500

In [6]:
aggregator = CandleAggregator(("1s", "1m"))
candles = []
for tick in ticks:
    candles.extend(aggregator.add_tick(tick))

# Flush the still-open final bucket per symbol/interval so the run closes out cleanly.
for stock in STOCKS:
    for interval in aggregator.intervals:
        final_candle = aggregator.snapshot(stock.symbol, interval)
        if final_candle is not None:
            candles.append(final_candle)

len(candles)

1530

In [7]:
TREND_VALUES = {trend.value for trend in MarketTrend}


def _trend_for_symbol(symbol: str) -> MarketTrend:
    """Behaviors outside MarketTrend's 4 values (e.g. volatility_spike) leave trend as normal."""
    matching = next((cfg for cfg in BEHAVIOR_CONFIGS if cfg.symbol == symbol), None)
    if matching is not None and matching.behavior_type.value in TREND_VALUES:
        return MarketTrend(matching.behavior_type.value)
    return MarketTrend.NORMAL


symbol_ticks: dict[str, list] = {}
for tick in ticks:
    symbol_ticks.setdefault(tick.symbol, []).append(tick)

market_states = []
for stock in STOCKS:
    stock_ticks = symbol_ticks[stock.symbol]
    first_price, last_price = stock_ticks[0].price, stock_ticks[-1].price
    momentum = float(max(Decimal("-1"), min(Decimal("1"), (last_price - first_price) / first_price)))
    market_states.append(
        MarketState(
            symbol=stock.symbol,
            trend=_trend_for_symbol(stock.symbol),
            volatility=stock.base_volatility,
            liquidity=0.6,
            momentum=momentum,
        )
    )

[(state.symbol, state.trend.value, round(state.momentum, 4)) for state in market_states]

[('ACME', 'uptrend', 0.0),
 ('BLTV', 'normal', 0.0001),
 ('CRSN', 'normal', -0.0001),
 ('DPHM', 'normal', -0.0001),
 ('EVRN', 'normal', -0.0003)]

In [8]:
SESSION_STARTED_AT = START_TIME
SESSION_ENDED_AT = START_TIME + (STEP_SIZE * STEPS)
SESSION_STATUS = "COMPLETED"

(SESSION_STARTED_AT.isoformat(), SESSION_ENDED_AT.isoformat())

('2026-01-02T08:30:00-06:00', '2026-01-02T08:35:00-06:00')

## 5. Validation

Assert the constraints enforced by the migrations (`bid < ask`, `bid <= price
<= ask`, OHLC bounds, uniqueness of sequence numbers / quote and candle keys)
before writing anything to disk.

In [9]:
assert len(ticks) == STEPS * len(STOCKS), "expected one tick per stock per step"

sequence_numbers = [tick.sequence_number for tick in ticks]
assert len(set(sequence_numbers)) == len(sequence_numbers), "sequence numbers must be unique"
assert sequence_numbers == sorted(sequence_numbers), "sequence numbers must be ascending"

for tick in ticks:
    assert tick.bid < tick.ask, f"tick bid/ask crossed for {tick.symbol}@{tick.timestamp}"
    assert tick.bid <= tick.price <= tick.ask, f"tick price outside bid/ask for {tick.symbol}@{tick.timestamp}"

for quote in quotes:
    assert quote.bid < quote.ask, f"quote bid/ask crossed for {quote.symbol}@{quote.timestamp}"

quote_keys = {(q.symbol, q.timestamp) for q in quotes}
assert len(quote_keys) == len(quotes), "quotes must be unique per (symbol, timestamp)"

for candle in candles:
    ohlc = (candle.open, candle.high, candle.low, candle.close)
    assert candle.low <= candle.high, f"candle low>high for {candle.symbol} {candle.interval}@{candle.timestamp}"
    assert candle.high == max(ohlc), f"candle high not max(OHLC) for {candle.symbol} {candle.interval}@{candle.timestamp}"
    assert candle.low == min(ohlc), f"candle low not min(OHLC) for {candle.symbol} {candle.interval}@{candle.timestamp}"

candle_keys = {(c.symbol, c.interval, c.timestamp) for c in candles}
assert len(candle_keys) == len(candles), "candles must be unique per (symbol, interval, timestamp)"

assert SESSION_ENDED_AT >= SESSION_STARTED_AT, "session ended_at must not precede started_at"

print("All validations passed:", len(ticks), "ticks,", len(quotes), "quotes,", len(candles), "candles.")

All validations passed: 1500 ticks, 1500 quotes, 1530 candles.


## 6. SQL serialization

Small escaping helpers, then one `INSERT` statement per table (batched into
chunks of 500 rows for the high-volume tables) written to
`migrations/seed_mock_market_data.sql`, in FK dependency order and wrapped in
a transaction.

In [10]:
def sql_str(value: str) -> str:
    return "'" + value.replace("'", "''") + "'"


def sql_ts(value: datetime) -> str:
    return sql_str(value.astimezone(MARKET_TIMEZONE).isoformat())


def sql_num(value) -> str:
    return str(value)


def sql_jsonb(value: dict) -> str:
    return sql_str(json.dumps(value)) + "::jsonb"


def chunked(sequence, size):
    for start in range(0, len(sequence), size):
        yield sequence[start:start + size]


BATCH_SIZE = 500

In [11]:
stocks_values = ",\n    ".join(
    f"({sql_str(s.symbol)}, {sql_str(s.company_name)}, {sql_num(s.starting_price)}, "
    f"{sql_str(s.sector)}, {s.average_volume}, {sql_num(s.base_volatility)})"
    for s in STOCKS
)
stocks_sql = (
    "INSERT INTO stocks (symbol, company_name, starting_price, sector, average_volume, base_volatility) VALUES\n"
    f"    {stocks_values};"
)

session_sql = (
    "INSERT INTO simulation_sessions (id, seed, drift, config, config_version, status, started_at, ended_at) VALUES\n"
    f"    ({SESSION_ID}, {SEED}, {DRIFT}, {sql_jsonb(SESSION_CONFIG)}, 1, {sql_str(SESSION_STATUS)}, "
    f"{sql_ts(SESSION_STARTED_AT)}, {sql_ts(SESSION_ENDED_AT)});\n"
    "SELECT setval(pg_get_serial_sequence('simulation_sessions', 'id'), (SELECT MAX(id) FROM simulation_sessions));"
)

market_states_values = ",\n    ".join(
    f"({SESSION_ID}, {sql_str(state.symbol)}, {sql_str(state.trend.value)}, "
    f"{sql_num(state.volatility)}, {state.liquidity}, {state.momentum})"
    for state in market_states
)
market_states_sql = (
    "INSERT INTO market_states (session_id, symbol, trend, volatility, liquidity, momentum) VALUES\n"
    f"    {market_states_values};"
)

market_behaviors_values = ",\n    ".join(
    f"({SESSION_ID}, {sql_str(cfg.symbol)}, {sql_str(cfg.behavior_type.value)}, "
    f"{sql_ts(START_TIME)}, {cfg.duration.total_seconds()}, {cfg.strength})"
    for cfg in BEHAVIOR_CONFIGS
)
market_behaviors_sql = (
    "INSERT INTO market_behaviors (session_id, symbol, behavior_type, start_time, duration_seconds, strength) VALUES\n"
    f"    {market_behaviors_values};"
)

len(stocks_sql), len(session_sql), len(market_states_sql), len(market_behaviors_sql)

(496, 365, 410, 258)

In [12]:
quotes_sql_statements = []
for batch in chunked(quotes, BATCH_SIZE):
    values = ",\n    ".join(
        f"({SESSION_ID}, {sql_str(q.symbol)}, {sql_ts(q.timestamp)}, {sql_num(q.bid)}, "
        f"{sql_num(q.ask)}, {q.bid_size}, {q.ask_size})"
        for q in batch
    )
    quotes_sql_statements.append(
        "INSERT INTO quotes (session_id, symbol, \"timestamp\", bid, ask, bid_size, ask_size) VALUES\n"
        f"    {values};"
    )

ticks_sql_statements = []
for batch in chunked(ticks, BATCH_SIZE):
    values = ",\n    ".join(
        f"({SESSION_ID}, {sql_str(t.symbol)}, {sql_ts(t.timestamp)}, {sql_num(t.price)}, {sql_num(t.bid)}, "
        f"{sql_num(t.ask)}, {t.bid_size}, {t.ask_size}, {t.trade_volume}, {t.sequence_number})"
        for t in batch
    )
    ticks_sql_statements.append(
        "INSERT INTO market_ticks (session_id, symbol, \"timestamp\", price, bid, ask, bid_size, ask_size, "
        "trade_volume, sequence_number) VALUES\n"
        f"    {values};"
    )

candles_sql_statements = []
for batch in chunked(candles, BATCH_SIZE):
    values = ",\n    ".join(
        f"({SESSION_ID}, {sql_str(c.symbol)}, {sql_str(c.interval)}, {sql_ts(c.timestamp)}, "
        f"{sql_num(c.open)}, {sql_num(c.high)}, {sql_num(c.low)}, {sql_num(c.close)}, {c.volume}, {c.trade_count})"
        for c in batch
    )
    candles_sql_statements.append(
        "INSERT INTO candles (session_id, symbol, \"interval\", \"timestamp\", open, high, low, close, "
        "volume, trade_count) VALUES\n"
        f"    {values};"
    )

len(quotes_sql_statements), len(ticks_sql_statements), len(candles_sql_statements)

(3, 3, 4)

In [13]:
sql_sections = [
    "-- Deterministic mock market data seed (generated by notebooks/seed_data_generation.ipynb).",
    "-- FMS market-data tables only; no users/accounts/orders/instruments are seeded.",
    "BEGIN;",
    "",
    stocks_sql,
    "",
    session_sql,
    "",
    market_states_sql,
    "",
    market_behaviors_sql,
    "",
    *quotes_sql_statements,
    "",
    *ticks_sql_statements,
    "",
    *candles_sql_statements,
    "",
    "COMMIT;",
    "",
]

OUTPUT_SQL_PATH.parent.mkdir(parents=True, exist_ok=True)
OUTPUT_SQL_PATH.write_text("\n".join(sql_sections), encoding="utf-8")
print(f"Wrote {OUTPUT_SQL_PATH} ({OUTPUT_SQL_PATH.stat().st_size:,} bytes)")

Wrote c:\Users\chris\Downloads\Github Desktop\FMS\Simulated_Engine\notebooks\data_seed\migrations\seed_mock_market_data.sql (411,015 bytes)


## 7. Reload and verify integrity

Re-read the written file from disk and confirm the expected number of
`INSERT` statements per table are present.

In [14]:
written_sql = OUTPUT_SQL_PATH.read_text(encoding="utf-8")

assert written_sql.count("INSERT INTO stocks") == 1
assert written_sql.count("INSERT INTO simulation_sessions") == 1
assert written_sql.count("INSERT INTO market_states") == 1
assert written_sql.count("INSERT INTO market_behaviors") == 1
assert written_sql.count("INSERT INTO quotes") == len(quotes_sql_statements)
assert written_sql.count("INSERT INTO market_ticks") == len(ticks_sql_statements)
assert written_sql.count("INSERT INTO candles") == len(candles_sql_statements)
assert written_sql.strip().startswith("--")
assert written_sql.strip().endswith("COMMIT;")

print("Seed file re-read from disk and section counts verified.")

Seed file re-read from disk and section counts verified.


## Applying the seed file

Run the numbered schema migrations first (see `migrations/README.md`), then
apply this file the same way:

```powershell
psql $env:DATABASE_URL -v ON_ERROR_STOP=1 -f "migrations/seed_mock_market_data.sql"
```